# Phase 2: High-Frequency 15T Data Processing
Load raw feeds.csv, clean, impute locally, resample to 15T, and save feeds_cleaned_15T.csv.

In [ ]:
import pandas as pd
import numpy as np
import os

# Load raw data
FILE_NAME = 'feeds.csv'
print(f"Loading {FILE_NAME}...")
df = pd.read_csv(FILE_NAME)
print(f"Loaded shape: {df.shape}")

# Convert created_at to datetime and set as index
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce', utc=True)
df = df.set_index('created_at').sort_index()
print(f"Index range: {df.index.min()} to {df.index.max()}")

# Rename columns
rename_map = {'field1': 'pm2_5', 'field2': 'pm10', 'field3': 'temperature', 'field4': 'humidity'}
df = df.rename(columns=rename_map)
print(f"Columns: {df.columns.tolist()}")

# Drop irrelevant columns
cols_to_drop = ['entry_id', 'latitude', 'longitude', 'elevation', 'status']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Convert sensor columns to numeric
sensor_cols = ['pm2_5', 'pm10', 'temperature', 'humidity']
for col in sensor_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Handle outliers (sensor error codes)
for col in sensor_cols:
    if col in df.columns:
        if 'pm' in col:
            threshold = 500
        elif col == 'humidity':
            threshold = 100
        elif col == 'temperature':
            threshold = 50
        else:
            threshold = np.inf
        bad = df[col] > threshold
        if bad.any():
            df.loc[bad, col] = np.nan
            print(f"Set {bad.sum()} outliers in {col} to NaN")

# Local imputation: 10-point rolling window
window_size = 10
for col in sensor_cols:
    if col in df.columns:
        rolling_mean = df[col].rolling(window=window_size, center=True, min_periods=1).mean()
        df[col] = df[col].fillna(rolling_mean)
        df[col] = df[col].fillna(method='ffill').fillna(method='bfill')

print(f"Missing values after imputation: {df[sensor_cols].isnull().sum().sum()}")

# Resample to 15T
df_15T = df[sensor_cols].resample('15T').mean()
print(f"Resampled to 15T: {df_15T.shape}")

# Impute any remaining NaNs in 15T data
for col in sensor_cols:
    if col in df_15T.columns:
        df_15T[col] = df_15T[col].interpolate(method='linear').ffill().bfill()

print(f"Missing values after 15T imputation: {df_15T.isnull().sum().sum()}")

# Save
df_15T.to_csv('feeds_cleaned_15T.csv')
print("Saved feeds_cleaned_15T.csv")

# Verify
print(f"Shape: {df_15T.shape}")
print("Head:")
print(df_15T.head())

Loading feeds.csv...


C:\Users\burak\AppData\Local\Temp\ipykernel_13548\4110174421.py:8: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(FILE_NAME)


Loaded shape: (291472, 10)
Index range: 2025-03-05 03:37:33+00:00 to 2025-07-30 05:10:54+00:00
Columns: ['entry_id', 'pm2_5', 'pm10', 'temperature', 'humidity', 'latitude', 'longitude', 'elevation', 'status']
Set 3904 outliers in temperature to NaN
Set 1056 outliers in humidity to NaN
Missing values after imputation: 0


C:\Users\burak\AppData\Local\Temp\ipykernel_13548\4110174421.py:53: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
C:\Users\burak\AppData\Local\Temp\ipykernel_13548\4110174421.py:58: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_15T = df[sensor_cols].resample('15T').mean()


Resampled to 15T: (14119, 4)
Saved feeds_cleaned_15T.csv
Shape: (14119, 4)
Head:
                               pm2_5       pm10  temperature   humidity
created_at                                                             
2025-03-05 03:30:00+00:00  32.629843  33.091489    23.916111  56.265000
2025-03-05 03:45:00+00:00  32.598134  31.212410    27.489744  61.546154
2025-03-05 04:00:00+00:00  31.862313  31.280222    22.385714  56.466667
2025-03-05 04:15:00+00:00  30.345942  32.086658    15.280000  48.600000
2025-03-05 04:30:00+00:00  31.162133  33.801427    12.833333  37.533333
